# 1.2 · 多表 JOIN / SQL Joins

> **课程定位 / Where this fits**
> **Part 1 第 2 课**。1.1 教了单表 SQL，本节把**多张表关联起来查**——SQL 真正的威力所在。
> **Part 1, lesson 2.** 1.1 covered single-table SQL; this lesson joins multiple tables — where SQL truly shines.

> 📐 **符号约定 / Notation**
> 表之间用 `→` 标外键方向：`album.artist_id → artist.artist_id` 表示 `album` 的 `artist_id` 指向 `artist` 表的主键。
> Arrow direction = FK reference.

> 💡 **面试相关 / Interview-relevant**
> - "解释 INNER vs LEFT JOIN" ★★★★★（必考）
> - "重复 key 导致行数爆炸" ★★★★★（陷阱题，工作里也常踩）
> - "JOIN 时 ON vs WHERE 的差别" ★★★★（特别是 LEFT JOIN）
> - "SQL 实现 anti-join（找出 A 表里 B 没有的）" ★★★★
> - "SELF JOIN 找员工的经理" ★★★
>
> Top hits: INNER vs LEFT, multiplication gotcha, ON vs WHERE, anti-join, self-join.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 准确说出 **INNER / LEFT / RIGHT / FULL / CROSS / SELF** 六种 JOIN 各自返回什么。
   Define what each of the six JOIN flavors returns.
2. 用 Venn 图直觉 + 实际查询**双轨**理解 JOIN。
   Reason about JOINs both with Venn diagrams and with concrete queries.
3. 解释 **`ON` vs `WHERE`** 在 LEFT JOIN 里截然不同的语义。
   Explain why `ON` and `WHERE` mean different things in LEFT JOIN.
4. 写出**反连接** / **半连接**（"找出 A 里 B 没有的"）的三种等价写法。
   Write anti-join / semi-join three ways.
5. 调试 **JOIN 后行数爆炸**（重复 key 的"笛卡尔积"陷阱）。
   Debug the multiplication-after-join gotcha.
6. 在 Mini Music Store 上**连查 3-4 张表**回答真实业务问题。
   Join 3-4 tables to answer real business questions.

---

## 目录 / Table of Contents

1. [JOIN 基础概念 / JOIN 101](#1)
2. [六种 JOIN 一图看懂 / Six Joins at a Glance](#2)
3. [INNER JOIN ⭐](#3)
4. [LEFT JOIN ⭐](#4)
5. [RIGHT JOIN & FULL OUTER JOIN](#5)
6. [`ON` vs `WHERE` —— LEFT JOIN 的灵魂区别](#6)
7. [`USING` 简化语法](#7)
8. [CROSS JOIN（笛卡尔积）](#8)
9. [SELF JOIN](#9)
10. [3 张表及以上的 JOIN / Multi-Table Joins](#10)
11. [⚠ 重复 key → 行数爆炸 / Multiplication Gotcha](#11)
12. [Anti-Join & Semi-Join](#12)
13. [实战：业务问题 8 连击 / Hands-on](#13)
14. [小结 / Summary](#14)


<a id="1"></a>
## 1. JOIN 基础概念 / JOIN 101

**为什么需要 JOIN？** 数据库做"范式化"——同一个信息**只存一份**，避免冗余。但查的时候要把它"拼回来"——这就是 JOIN。
**Why JOIN?** Databases normalize — each fact lives in one place. JOINs glue them back together for queries.

例：`album` 表只存 `artist_id`，要看艺术家名字必须 JOIN `artist` 表。
Example: `album` only stores `artist_id`; to see the artist's name, JOIN with `artist`.

### JOIN 的基本形式 / Basic syntax

```sql
SELECT <columns>
FROM   <left_table>
JOIN   <right_table>  ON <join_condition>;
```

`<join_condition>` 一般是"两边的 key 相等"：
The condition is usually "keys match":

```sql
FROM album AS a
JOIN artist AS ar ON a.artist_id = ar.artist_id
```

> 💡 **别名 / Aliases**：JOIN 多表时**必须用别名**让代码可读，列名重名时也必须用别名消歧义。
> **Always alias** tables in multi-table JOINs for readability and to disambiguate columns.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

print(f"duckdb : {duckdb.__version__}")


In [ ]:
# 重新构建 Mini Music Store（和 1.1 节相同）/ Rebuild dataset from 1.1
conn = duckdb.connect()

conn.sql("""
CREATE TABLE artist (
    artist_id INTEGER PRIMARY KEY,
    name      VARCHAR,
    country   VARCHAR
);
INSERT INTO artist VALUES
    (1, 'The Beatles',     'UK'),
    (2, 'Pink Floyd',      'UK'),
    (3, 'Miles Davis',     'US'),
    (4, 'Daft Punk',       'FR'),
    (5, 'Radiohead',       'UK'),
    (6, 'Anonymous Artist', NULL);

CREATE TABLE album (
    album_id  INTEGER PRIMARY KEY,
    title     VARCHAR,
    artist_id INTEGER,
    year      INTEGER
);
INSERT INTO album VALUES
    (1, 'Abbey Road',                    1, 1969),
    (2, 'The Dark Side of the Moon',     2, 1973),
    (3, 'The Wall',                      2, 1979),
    (4, 'Kind of Blue',                  3, 1959),
    (5, 'Discovery',                     4, 2001),
    (6, 'Random Access Memories',        4, 2013),
    (7, 'OK Computer',                   5, 1997),
    (8, 'Demos (unreleased)',            6, 2024);

CREATE TABLE track (
    track_id  INTEGER PRIMARY KEY,
    name      VARCHAR,
    album_id  INTEGER,
    genre     VARCHAR,
    seconds   INTEGER,
    price     DECIMAL(4, 2)
);
INSERT INTO track VALUES
    (1,  'Come Together',           1, 'Rock',       259, 0.99),
    (2,  'Something',                1, 'Rock',       182, 0.99),
    (3,  'Here Comes the Sun',       1, 'Rock',       185, 0.99),
    (4,  'Time',                     2, 'Rock',       413, 1.29),
    (5,  'Money',                    2, 'Rock',       382, 1.29),
    (6,  'Us and Them',              2, 'Rock',       460, 1.29),
    (7,  'Another Brick in the Wall',3, 'Rock',       239, 1.29),
    (8,  'Comfortably Numb',         3, 'Rock',       382, 1.29),
    (9,  'So What',                  4, 'Jazz',       545, 1.49),
    (10, 'Freddie Freeloader',       4, 'Jazz',       586, 1.49),
    (11, 'Blue in Green',            4, 'Jazz',       337, 1.49),
    (12, 'One More Time',            5, 'Electronic', 320, 1.29),
    (13, 'Aerodynamic',              5, 'Electronic', 213, 1.29),
    (14, 'Digital Love',             5, 'Electronic', 301, 1.29),
    (15, 'Get Lucky',                6, 'Electronic', 369, 1.29),
    (16, 'Instant Crush',            6, 'Electronic', 337, 1.29),
    (17, 'Lose Yourself to Dance',   6, 'Electronic', 353, 1.29),
    (18, 'Paranoid Android',         7, 'Rock',       384, 1.29),
    (19, 'Karma Police',             7, 'Rock',       261, 1.29),
    (20, 'No Surprises',             7, 'Rock',       228, 1.29),
    (21, 'Untitled Demo 1',          8, 'Rock',       180, 0.50),
    (22, 'Untitled Demo 2',          8, 'Rock',       195, 0.50);

CREATE TABLE customer (
    customer_id INTEGER PRIMARY KEY,
    name        VARCHAR,
    country     VARCHAR,
    email       VARCHAR
);
INSERT INTO customer VALUES
    (1, 'Alice Chen',    'US', 'alice@example.com'),
    (2, 'Bob Smith',     'UK', 'BOB@example.com'),
    (3, 'Charlie Davis', 'US', 'charlie@example.com'),
    (4, 'Diana Park',    'DE', 'diana@example.com'),
    (5, 'Ethan Miller',  'US', 'ethan@example.com'),
    (6, 'Fiona Wong',    'JP', NULL);

CREATE TABLE invoice (
    invoice_id   INTEGER PRIMARY KEY,
    customer_id  INTEGER,
    track_id     INTEGER,
    invoice_date DATE,
    quantity     INTEGER
);
INSERT INTO invoice VALUES
    (1,  1, 1,  DATE '2026-01-05', 1),
    (2,  1, 9,  DATE '2026-01-05', 2),
    (3,  2, 4,  DATE '2026-01-10', 1),
    (4,  2, 18, DATE '2026-01-10', 1),
    (5,  3, 5,  DATE '2026-02-12', 1),
    (6,  3, 6,  DATE '2026-02-12', 1),
    (7,  3, 7,  DATE '2026-02-12', 3),
    (8,  4, 15, DATE '2026-02-20', 1),
    (9,  4, 12, DATE '2026-02-20', 1),
    (10, 4, 13, DATE '2026-02-20', 1),
    (11, 5, 9,  DATE '2026-03-01', 1),
    (12, 5, 10, DATE '2026-03-01', 1),
    (13, 5, 11, DATE '2026-03-01', 1),
    (14, 5, 4,  DATE '2026-03-05', 2),
    (15, 1, 18, DATE '2026-03-15', 1),
    (16, 1, 19, DATE '2026-03-15', 1),
    (17, 2, 15, DATE '2026-04-01', 1),
    (18, 3, 14, DATE '2026-04-10', 2),
    (19, 4, 8,  DATE '2026-05-02', 1),
    (20, 5, 1,  DATE '2026-05-20', 1);
""")

print("Tables ready:")
print(conn.sql("SHOW TABLES").df())


<a id="2"></a>
## 2. 六种 JOIN 一图看懂 / Six Joins at a Glance

```
left ⨝ right    INNER       LEFT       RIGHT       FULL       CROSS       SELF
                ┌─┬─┐       ┌─┬─┐      ┌─┬─┐       ┌─┬─┐      L × R         同表对自己 JOIN
                │ │L│       │L│L│      │ │L│       │L│L│
                ├─┼─┤       ├─┼─┤      ├─┼─┤       ├─┼─┤
                │R│✓│       │R│✓│      │R│✓│       │R│✓│
                ├─┼─┤       ├─┼─┤      ├─┼─┤       ├─┼─┤
                │ │ │       │ │L│      │R│ │       │L│R│
                └─┴─┘       └─┴─┘      └─┴─┘       └─┴─┘
                只匹配      左全留      右全留       两边全留
```

| Join Type | 返回 / Returns |
|---|---|
| **INNER**  | 只返回**两边匹配上**的行 / only matching rows from both |
| **LEFT**   | 左表**全保留**；右表没匹配 → NULL |
| **RIGHT**  | 镜像 LEFT；很少用，可以改写成 LEFT |
| **FULL**   | 两边都保留；未匹配位置 → NULL |
| **CROSS**  | 笛卡尔积 / Cartesian product = $\|L\| \times \|R\|$ 行 |
| **SELF**   | 一张表"自己 JOIN 自己"（用别名区分）|

下面用 Venn 图把它们可视化：
Let's visualize with Venn diagrams:


In [ ]:
# 画 6 种 JOIN 的 Venn 图直觉 / Venn-style schematic for 6 JOINs
from matplotlib.patches import Circle, Rectangle

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
join_types = [
    ("INNER", ("none", "lightblue", "none")),    # only intersection
    ("LEFT",  ("lightblue", "lightblue", "none")),
    ("RIGHT", ("none", "lightblue", "lightblue")),
    ("FULL",  ("lightblue", "lightblue", "lightblue")),
    ("CROSS", ("salmon", "salmon", "salmon")),   # all combinations
    ("SELF",  ("plum", "plum", "none")),         # same table joined to itself
]

for ax, (name, (l, mid, r)) in zip(axes, join_types):
    if name == "CROSS":
        ax.add_patch(Rectangle((-1.5, -1), 3, 2, color=mid, alpha=0.6))
        ax.text(0, 0, "L × R", ha="center", va="center", fontsize=14, fontweight="bold")
    elif name == "SELF":
        ax.add_patch(Circle((-0.4, 0), 0.9, color=l, alpha=0.55, ec="black"))
        ax.add_patch(Circle((0.4, 0), 0.9, color=r if r != "none" else "white", alpha=0.55, ec="black"))
        ax.text(0, -1.3, "same table\nas L & R", ha="center", fontsize=9)
    else:
        # Left circle
        ax.add_patch(Circle((-0.4, 0), 0.9, color=l if l != "none" else "white", alpha=0.5, ec="black"))
        # Right circle
        ax.add_patch(Circle((0.4, 0), 0.9, color=r if r != "none" else "white", alpha=0.5, ec="black"))
        # Intersection
        ax.add_patch(Circle((0, 0), 0.4, color=mid if mid != "none" else "white", alpha=0.7, ec="black"))
        ax.text(-0.8, 0, "L", ha="center", va="center", fontsize=12, fontweight="bold")
        ax.text(0.8, 0, "R", ha="center", va="center", fontsize=12, fontweight="bold")
    ax.set_xlim(-2, 2); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(name, fontsize=13, fontweight="bold")

plt.suptitle("The Six Joins — visual cheat sheet", y=1.05, fontsize=14)
plt.tight_layout()
plt.show()


<a id="3"></a>
## 3. INNER JOIN ⭐ —— 最常用的 JOIN

**只返回两边匹配上的行**。SQL 里直接写 `JOIN` 默认就是 `INNER JOIN`。
**Returns only matching rows.** Bare `JOIN` = `INNER JOIN`.

```sql
SELECT ...
FROM   L
INNER JOIN R ON L.key = R.key;
```


In [ ]:
# 每张专辑 + 它的艺术家名字 / Each album + its artist's name
conn.sql("""
    SELECT
        a.title       AS album_title,
        a.year        AS album_year,
        ar.name       AS artist_name,
        ar.country    AS artist_country
    FROM album AS a
    INNER JOIN artist AS ar
        ON a.artist_id = ar.artist_id
    ORDER BY a.year;
""").df()


**全部 8 张专辑都出现了** —— 因为每张专辑都能匹配到一个 artist。
All 8 albums appear — every album has a matching artist.

下面我们故意把一张"孤立"的专辑加进来，看 INNER 怎么"吃掉"它：
Now let's add an orphan album to see INNER silently drop it:


In [ ]:
# 加一张 artist_id 不存在的专辑 / Add an orphan album (no matching artist)
conn.sql("""
    INSERT INTO album VALUES
    (99, 'Orphan Album (no artist)', 999, 2025);
""")

# INNER JOIN 看不到 orphan / Orphan disappears with INNER
print("INNER JOIN result:")
print(conn.sql("""
    SELECT a.title, ar.name AS artist
    FROM album AS a
    INNER JOIN artist AS ar ON a.artist_id = ar.artist_id;
""").df())

print(f"\nrows in album: {conn.sql('SELECT COUNT(*) FROM album').fetchone()[0]}")
print("rows after INNER JOIN with artist:",
      conn.sql("""SELECT COUNT(*) FROM album a INNER JOIN artist ar
                   ON a.artist_id = ar.artist_id""").fetchone()[0])


**注意**：album 表有 9 行，但 INNER JOIN 只返回 8 行 —— **`Orphan Album` 被悄悄"吃掉"了**。这正是 INNER JOIN 的语义。
**Important**: 9 rows in album, INNER JOIN returns 8 — the orphan is silently dropped. That's INNER's semantics.

如果你想"**保留左表所有行**"——用 LEFT JOIN。
If you need to keep all left-table rows, use LEFT JOIN.


<a id="4"></a>
## 4. LEFT JOIN ⭐ —— "保留左表所有行"

```sql
SELECT ...
FROM   L
LEFT JOIN R ON L.key = R.key;
```

- 左表**所有行**都出现
- 右表能匹配 → 正常拼上
- 右表**没匹配 → 右侧列都填 NULL**


In [ ]:
# 现在用 LEFT JOIN：orphan 也会出现，artist 列为 NULL
# LEFT JOIN keeps the orphan; artist columns become NULL
conn.sql("""
    SELECT
        a.title        AS album_title,
        a.year         AS album_year,
        ar.name        AS artist_name,
        ar.country     AS artist_country
    FROM album AS a
    LEFT JOIN artist AS ar
        ON a.artist_id = ar.artist_id
    ORDER BY a.year;
""").df()


**`Orphan Album` 出现了**，但右侧 artist 列是 `None` / `NaN`。这就是 LEFT JOIN 的价值——**显式承认"右表没匹配"**而不是默默丢数据。
The orphan appears with NULL artist columns — LEFT JOIN explicitly admits "no match" rather than silently dropping.

### LEFT JOIN 的典型工业用法 / Industrial uses

1. **保留所有 user 的查询**：`users LEFT JOIN purchases` → 没买过东西的用户也保留（计算转化率必备）
2. **属性挂载**：`events LEFT JOIN user_profile` → 没注册资料的事件也保留
3. **检测孤儿数据**：`A LEFT JOIN B WHERE B.key IS NULL` → 找出 A 里 B 没有的（anti-join）


In [ ]:
# 应用 1：每个艺术家有多少张专辑（含 0 张的）
# Use case 1: album count per artist, INCLUDING those with 0 albums
conn.sql("""
    SELECT
        ar.name                  AS artist,
        COUNT(a.album_id)        AS n_albums
    FROM artist AS ar
    LEFT JOIN album AS a
        ON ar.artist_id = a.artist_id
    GROUP BY ar.name
    ORDER BY n_albums DESC;
""").df()


**注意 `COUNT(a.album_id)` vs `COUNT(*)`**：
- `COUNT(a.album_id)` 忽略 NULL → "没专辑的艺术家"显示 0 ✅
- `COUNT(*)` 即使没匹配也算 1 → "没专辑的艺术家"显示 1 ❌

**Note** `COUNT(a.album_id)` (NULL-aware) gives 0 for artists with no albums; `COUNT(*)` would lie and say 1.

> ⚠ 这是 **面试常考的细节坑**。
> A frequent interview catch.


<a id="5"></a>
## 5. RIGHT JOIN & FULL OUTER JOIN

### RIGHT JOIN

镜像 LEFT JOIN —— 右表全保留。**实际工作中几乎不用**——可以**永远写成 LEFT**（把两张表位置换一下）。
Mirror of LEFT — keeps all right rows. **Rarely used in practice** — always rewrite as LEFT by swapping table order.

```sql
A RIGHT JOIN B  ≡  B LEFT JOIN A
```

### FULL OUTER JOIN

**两边都保留**——左右无匹配的位置都填 NULL。
Keep all rows from both sides; fill NULL where no match.

```sql
SELECT ...
FROM L
FULL OUTER JOIN R ON L.key = R.key;
```


In [ ]:
# 加一个孤立的 artist 进 artist 表 / Add an orphan artist (no albums)
conn.sql("""
    INSERT INTO artist VALUES
    (100, 'Lonely Artist (no albums)', 'CA');
""")

# FULL OUTER JOIN：orphan album 和 orphan artist 都保留
# FULL OUTER JOIN keeps both orphans
result = conn.sql("""
    SELECT
        a.title         AS album_title,
        ar.name         AS artist_name
    FROM album AS a
    FULL OUTER JOIN artist AS ar
        ON a.artist_id = ar.artist_id
    ORDER BY ar.name NULLS LAST, a.title NULLS LAST;
""").df()
print(result)


**两个孤儿都出现了** —— `Orphan Album` 的 artist 是 NULL，`Lonely Artist` 的 album 是 NULL。
Both orphans appear: Orphan Album with NULL artist, Lonely Artist with NULL album.

### 应用：找数据"两边对不齐"的地方

```sql
FULL OUTER JOIN ...
WHERE L.key IS NULL OR R.key IS NULL;
```

这个组合**专门找两表 key 对不齐**的所有行——数据迁移、ETL 对账常用。
This pattern surfaces every mismatched key — common in ETL reconciliation.


<a id="6"></a>
## 6. `ON` vs `WHERE` —— LEFT JOIN 的灵魂区别

**面试 ★★★★★** —— 太多人在这里栽跟头。
**The #1 interview trap.**

```sql
-- (A) 过滤条件放 ON
SELECT ... FROM L LEFT JOIN R
  ON L.key = R.key AND R.status = 'active';

-- (B) 过滤条件放 WHERE
SELECT ... FROM L LEFT JOIN R
  ON L.key = R.key
WHERE R.status = 'active';
```

**它俩结果不一样！** Why?

- **(A) `ON` 里加条件**：在 JOIN 时筛选**右表的匹配项**。左表"没匹配"的行依然保留（右侧 NULL）。
- **(B) `WHERE` 里加条件**：JOIN 完之后再过滤。但 LEFT JOIN 产生的 NULL 行被 `R.status = 'active'` **筛掉了**——实质上变成了 INNER JOIN！

**(A) keeps unmatched left rows; (B) silently turns LEFT into INNER** because `NULL = 'active'` is NULL (not TRUE).


In [ ]:
# 演示：所有艺术家 + 1973 年的专辑 / Each artist + their 1973 album
# Version A: 条件放 ON / Condition in ON
print("--- A) condition in ON: keep all artists ---")
print(conn.sql("""
    SELECT ar.name AS artist, a.title AS album_1973
    FROM artist AS ar
    LEFT JOIN album AS a
      ON ar.artist_id = a.artist_id AND a.year = 1973
    ORDER BY ar.name;
""").df())

# Version B: 条件放 WHERE / Condition in WHERE → silently becomes INNER
print("\n--- B) condition in WHERE: silently becomes INNER ---")
print(conn.sql("""
    SELECT ar.name AS artist, a.title AS album_1973
    FROM artist AS ar
    LEFT JOIN album AS a
      ON ar.artist_id = a.artist_id
    WHERE a.year = 1973
    ORDER BY ar.name;
""").df())


**对比**：
- (A) 每个艺术家都出现，没有 1973 年专辑的显示 NULL（保留意图：所有艺术家都要）
- (B) 只剩 Pink Floyd 一行——因为其他艺术家在 1973 年没专辑，被 `WHERE a.year = 1973` 过滤掉了
  Only Pink Floyd appears — others get filtered because their `a.year` is NULL.

### 一句话规则 / The rule

> **左表本身的过滤** → 放 `WHERE` ✅
> **右表的过滤（且想保留左表 unmatched 行）** → 放 `ON` ✅


<a id="7"></a>
## 7. `USING` 简化语法 / `USING` shortcut

如果两张表 JOIN 的列**同名**，可以用 `USING(col)`：
When join columns share a name, `USING(col)` is shorter:

```sql
-- 这两行等价 / These are equivalent
... ON L.artist_id = R.artist_id
... USING (artist_id)
```

`USING` 还会**自动合并**两表的同名列为一个（不会出现两个 `artist_id`）。
`USING` also dedupes the join column in the output.


In [ ]:
conn.sql("""
    SELECT
        a.title,
        ar.name AS artist
    FROM album AS a
    JOIN artist AS ar USING (artist_id)   -- 简化 / shortened
    LIMIT 5;
""").df()


**注意**：实际工作中 `USING` 不常见——明确写 `ON L.col = R.col` 可读性更好。**JOIN 表多了之后**（3+ 张）`USING` 反而容易混乱。
In practice explicit `ON L.col = R.col` is more readable; `USING` gets confusing with 3+ tables.


<a id="8"></a>
## 8. CROSS JOIN（笛卡尔积）/ Cartesian Product

```sql
SELECT ... FROM L CROSS JOIN R;
-- 或省略关键字 / or omit keyword:
SELECT ... FROM L, R;
```

返回 $\|L\| \times \|R\|$ 行——所有可能组合。
Returns $|L| \times |R|$ rows — every combination.

**应用 / Uses**：
- 生成笛卡尔表（"所有 (用户 × 日期) 对"用于补齐稀疏时间序列）
- 探索"理论上能买什么"
- 调试时**慎用**——10K × 10K = 100M 行 RAM 爆炸


In [ ]:
# 5 国家 × 4 种支付方式 = 所有可能组合 / All country × payment combos
country_df = pd.DataFrame({"country": ["US", "UK", "DE", "JP", "CA"]})
payment_df = pd.DataFrame({"payment": ["credit", "debit", "paypal", "applepay"]})

conn.sql("""
    SELECT country, payment
    FROM country_df
    CROSS JOIN payment_df
    ORDER BY country, payment;
""").df()


**5 × 4 = 20 行**——所有可能组合。
20 rows — every combination.

### 经典实战 / Classic real-world use

补全"每天每用户的活跃情况"：先做 `(date × user)` 的笛卡尔积，再 LEFT JOIN 实际事件——**没活跃的格子留 NULL 填 0**。
Fill in (date × user) gaps: build the cartesian, LEFT JOIN actual events, treat NULL as 0.


<a id="9"></a>
## 9. SELF JOIN —— 一张表自己 JOIN 自己

**用别名把同一张表当成两张用**。最经典的例子：员工表里"找每个员工的经理"。
**Two aliases for the same table.** Classic: find each employee's manager from a flat employee table.

我们用 album 表演示——"找出**同一年发行**的不同专辑配对"：
We'll use album to demo — "find pairs of albums released the same year":


In [ ]:
# 删掉前面加的孤儿数据，恢复干净状态 / Clean orphans for clearer demo
conn.sql("DELETE FROM album WHERE album_id = 99;")
conn.sql("DELETE FROM artist WHERE artist_id = 100;")

# Self-JOIN: 同年发行的不同专辑对 / Pairs of different albums released same year
conn.sql("""
    SELECT
        a1.title  AS album_a,
        a2.title  AS album_b,
        a1.year
    FROM album AS a1
    JOIN album AS a2
        ON a1.year = a2.year
       AND a1.album_id < a2.album_id      -- 避免 (A, B) 和 (B, A) 都出现，也排除自己 / dedupe pairs + exclude self
    ORDER BY a1.year;
""").df()


**注意 `a1.album_id < a2.album_id`** —— 不写这条会出现两个问题：
1. **重复**：`(A, B)` 和 `(B, A)` 都出现
2. **自己跟自己 join**：`(A, A)` 也会出现

`<` 一举解决两个问题。`<=` 也能去重但会保留自己。
Without `<`, you'd see both `(A, B)` and `(B, A)` and trivial `(A, A)`. Use `<` to eliminate both at once.


<a id="10"></a>
## 10. 3 张表及以上的 JOIN / Multi-Table Joins

**线性 chain** 直接堆 JOIN：
Linear chain — just stack JOINs:

```sql
SELECT ...
FROM    A
JOIN    B ON A.k = B.k
JOIN    C ON B.k2 = C.k2
JOIN    D ON C.k3 = D.k3;
```


In [ ]:
# track → album → artist 三表 JOIN / track → album → artist
conn.sql("""
    SELECT
        t.name        AS track,
        a.title       AS album,
        ar.name       AS artist,
        t.genre
    FROM track AS t
    JOIN album  AS a  ON t.album_id  = a.album_id
    JOIN artist AS ar ON a.artist_id = ar.artist_id
    ORDER BY ar.name, a.title, t.name
    LIMIT 10;
""").df()


In [ ]:
# 4 表 JOIN：每张发票 + 客户 + 歌曲 + 专辑 + 艺术家
# 4-table chain: invoices with full context
conn.sql("""
    SELECT
        i.invoice_id,
        i.invoice_date,
        c.name        AS customer,
        c.country     AS cust_country,
        t.name        AS track,
        ar.name       AS artist
    FROM invoice  AS i
    JOIN customer AS c  ON i.customer_id = c.customer_id
    JOIN track    AS t  ON i.track_id    = t.track_id
    JOIN album    AS a  ON t.album_id    = a.album_id
    JOIN artist   AS ar ON a.artist_id   = ar.artist_id
    ORDER BY i.invoice_date, i.invoice_id
    LIMIT 6;
""").df()


<a id="11"></a>
## 11. ⚠ 重复 key → 行数爆炸 / Multiplication Gotcha

**JOIN 最大的陷阱**：如果 JOIN key 在**一边或两边重复**，结果行数会"乘起来"。
**Biggest JOIN pitfall**: duplicate keys multiply the result.

数学上 / Mathematically：

$$\text{rows}_{\text{out}} = \sum_{k} n_L(k) \cdot n_R(k)$$

如果 $k$ 在 $L$ 出现 1 次、$R$ 出现 3 次 → 输出 3 行（看似无害）。
但如果 $L$ 出现 5 次、$R$ 出现 4 次 → **20 行**！很容易 SUM 出错。


In [ ]:
# 制造一个"重复 key"演示 / Make a table with duplicate keys
A = pd.DataFrame({
    "uid": [1, 1, 2, 3],
    "purchase_amt": [100, 50, 200, 300],
})
B = pd.DataFrame({
    "uid": [1, 1, 1, 2],
    "campaign": ["email1", "email2", "push", "email1"],
})

print("A:")
print(A)
print("\nB:")
print(B)


In [ ]:
# Naive JOIN: 行数爆炸 / Naive JOIN multiplies
result = conn.sql("""
    SELECT A.uid, A.purchase_amt, B.campaign
    FROM A JOIN B USING (uid)
""").df()
print("Joined rows:", len(result))
print(result)

# 每个 uid 的 purchase_amt 总和：被错误地乘了！
# Sum of purchase_amt by uid: WRONG! Each purchase appears multiple times
wrong_sum = conn.sql("""
    SELECT A.uid, SUM(A.purchase_amt) AS total
    FROM A JOIN B USING (uid)
    GROUP BY A.uid
""").df()
print("\n--- WRONG total per uid (multiplied) ---")
print(wrong_sum)

# 真实的总和 / Actual sum per uid
correct = A.groupby("uid")["purchase_amt"].sum().reset_index().rename(columns={"purchase_amt": "total"})
print("\n--- CORRECT total per uid ---")
print(correct)


**看 uid=1**：
- 真实购买总额 = 100 + 50 = **150**
- JOIN 后 SUM = 100×3 + 50×3 = **450**（因为 uid=1 在 B 里有 3 行）
- For uid=1, real total is 150 but JOINed SUM is 450 because uid=1 has 3 rows in B.

这是工业里**最容易做错的指标**之一。
This is one of the easiest metrics to get wrong in industry.

### 怎么避免 / How to avoid

1. **聚合后再 JOIN**（推荐 ⭐）：
   Aggregate before joining:
   ```sql
   FROM A
   JOIN (SELECT uid, COUNT(*) AS n_campaigns FROM B GROUP BY uid) AS Bagg
        USING (uid)
   ```
2. **JOIN 后 `DISTINCT`**（粗暴但有效）
3. **JOIN 前检查每张表的 key 唯一性**：`SELECT COUNT(*) - COUNT(DISTINCT uid) FROM ...`


In [ ]:
# 正确做法：先聚合 B，再 JOIN / Aggregate B first, then JOIN
correct = conn.sql("""
    SELECT
        A.uid,
        SUM(A.purchase_amt)         AS total_purchases,
        Bagg.n_campaigns
    FROM A
    JOIN (
        SELECT uid, COUNT(*) AS n_campaigns
        FROM B
        GROUP BY uid
    ) AS Bagg USING (uid)
    GROUP BY A.uid, Bagg.n_campaigns
    ORDER BY A.uid;
""").df()
print(correct)


<a id="12"></a>
## 12. Anti-Join & Semi-Join

**Anti-join**：找出"**A 里 B 没有的**"。
"Find A rows with no match in B."

**Semi-join**：找出"**A 里 B 有的**"——只要"存在"不要展开。
"Find A rows that DO have a match — just existence, no expansion."

SQL 没有 `ANTI JOIN` 关键字（PostgreSQL 16+ 有 syntax 但不通用），实务里有 **3 种**等价写法：
SQL doesn't have an `ANTI JOIN` keyword in most dialects. Three equivalent ways:

### Anti-join 三连 / Three ways to anti-join

```sql
-- (1) NOT IN
SELECT * FROM A WHERE A.k NOT IN (SELECT k FROM B);

-- (2) NOT EXISTS  ★ 通常最快 / usually fastest
SELECT * FROM A
WHERE NOT EXISTS (SELECT 1 FROM B WHERE B.k = A.k);

-- (3) LEFT JOIN + IS NULL
SELECT A.* FROM A
LEFT JOIN B ON A.k = B.k
WHERE B.k IS NULL;
```

> ⚠ **`NOT IN` 的坑**：如果右边的子查询**返回任何 NULL**，整个 `NOT IN` 永远返回 0 行！原因：`x NOT IN (1, NULL)` 等价于 `x != 1 AND x != NULL`，后者是 NULL → 整个表达式 NULL。
> **NOT IN gotcha**: if the subquery returns any NULL, `NOT IN` returns nothing! Because `x != NULL` is NULL, not TRUE.
>
> 工业实践：**优先用 `NOT EXISTS`**。
> Industry pref: `NOT EXISTS` over `NOT IN`.


In [ ]:
# 业务问题：从未买过任何东西的客户 / Customers who never purchased
# 三种等价写法 / Three equivalent ways
print("--- (1) NOT IN ---")
print(conn.sql("""
    SELECT name FROM customer
    WHERE customer_id NOT IN (SELECT customer_id FROM invoice);
""").df())

print("\n--- (2) NOT EXISTS ---")
print(conn.sql("""
    SELECT c.name FROM customer AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM invoice AS i WHERE i.customer_id = c.customer_id
    );
""").df())

print("\n--- (3) LEFT JOIN + IS NULL ---")
print(conn.sql("""
    SELECT c.name
    FROM customer AS c
    LEFT JOIN invoice AS i USING (customer_id)
    WHERE i.customer_id IS NULL;
""").df())


三种写法**结果一致**（这里 6 个客户都买过东西，所以都是空表）。
All three produce the same result (here all 6 customers have purchased, so empty).

### Semi-join：用 `EXISTS` / `IN`

```sql
-- 买过东西的客户 / customers who DID purchase
SELECT * FROM customer
WHERE customer_id IN (SELECT customer_id FROM invoice);

-- 等价 / Equivalent
SELECT * FROM customer AS c
WHERE EXISTS (SELECT 1 FROM invoice WHERE customer_id = c.customer_id);
```

> 💡 **EXISTS vs JOIN 选哪个 / EXISTS vs JOIN**
> - 只想知道"有没有"，不要展开 → **EXISTS / IN**
> - 想看具体匹配上的信息 → **JOIN**


<a id="13"></a>
## 13. 实战：业务问题 8 连击 / 8 Business Questions

把这一节学的 JOIN 串起来。**先思考 SQL 怎么写，再展开答案**。
Try writing each SQL yourself first.


In [ ]:
# Q1: 每张专辑 + 它的艺术家国家 / Album with artist's country
conn.sql("""
    SELECT a.title, ar.country
    FROM album AS a
    JOIN artist AS ar ON a.artist_id = ar.artist_id
    ORDER BY ar.country, a.title;
""").df()


In [ ]:
# Q2: 没专辑的艺术家 / Artists with NO album (anti-join)
conn.sql("""
    SELECT ar.name
    FROM artist AS ar
    LEFT JOIN album AS a ON ar.artist_id = a.artist_id
    WHERE a.album_id IS NULL;
""").df()


In [ ]:
# Q3: 每个客户的总消费金额 / Total spend per customer
# 注意：用 quantity × price，需要 invoice → track JOIN
conn.sql("""
    SELECT
        c.name                                          AS customer,
        c.country,
        COUNT(DISTINCT i.invoice_id)                    AS n_invoices,
        ROUND(SUM(i.quantity * t.price), 2)             AS total_spend
    FROM customer AS c
    JOIN invoice  AS i ON c.customer_id = i.customer_id
    JOIN track    AS t ON i.track_id    = t.track_id
    GROUP BY c.customer_id, c.name, c.country
    ORDER BY total_spend DESC;
""").df()


In [ ]:
# Q4: 每种 genre 的总销售额 / Total revenue per genre
conn.sql("""
    SELECT
        t.genre,
        SUM(i.quantity)                       AS units_sold,
        ROUND(SUM(i.quantity * t.price), 2)   AS revenue
    FROM invoice AS i
    JOIN track   AS t USING (track_id)
    GROUP BY t.genre
    ORDER BY revenue DESC;
""").df()


In [ ]:
# Q5: 哪个艺术家最赚钱？/ Top artist by revenue
conn.sql("""
    SELECT
        ar.name                                AS artist,
        ROUND(SUM(i.quantity * t.price), 2)    AS revenue,
        COUNT(DISTINCT i.invoice_id)           AS n_orders
    FROM invoice AS i
    JOIN track   AS t  ON i.track_id  = t.track_id
    JOIN album   AS a  ON t.album_id  = a.album_id
    JOIN artist  AS ar ON a.artist_id = ar.artist_id
    GROUP BY ar.artist_id, ar.name
    ORDER BY revenue DESC;
""").df()


In [ ]:
# Q6: 同一天买过 2 首及以上不同歌的客户 / Customers who bought 2+ distinct tracks on the same day
conn.sql("""
    SELECT
        c.name,
        i.invoice_date,
        COUNT(DISTINCT i.track_id) AS distinct_tracks
    FROM invoice  AS i
    JOIN customer AS c USING (customer_id)
    GROUP BY c.customer_id, c.name, i.invoice_date
    HAVING COUNT(DISTINCT i.track_id) >= 2
    ORDER BY i.invoice_date;
""").df()


In [ ]:
# Q7: 用 SELF JOIN: 找在同一天买东西的客户对 / Pairs of customers who shopped on the same day
conn.sql("""
    SELECT DISTINCT
        c1.name        AS customer_a,
        c2.name        AS customer_b,
        i1.invoice_date
    FROM invoice  AS i1
    JOIN invoice  AS i2
      ON i1.invoice_date = i2.invoice_date
     AND i1.customer_id  < i2.customer_id     -- 去重 + 排除自己 / dedupe + exclude self
    JOIN customer AS c1 ON i1.customer_id = c1.customer_id
    JOIN customer AS c2 ON i2.customer_id = c2.customer_id
    ORDER BY i1.invoice_date;
""").df()


In [ ]:
# Q8: 每个客户买的"专辑数"（避免重复 key 引起的爆炸）
# Albums purchased per customer — careful with multiplication
conn.sql("""
    SELECT
        c.name,
        COUNT(DISTINCT a.album_id) AS distinct_albums_bought
    FROM customer AS c
    JOIN invoice  AS i USING (customer_id)
    JOIN track    AS t USING (track_id)
    JOIN album    AS a USING (album_id)
    GROUP BY c.customer_id, c.name
    ORDER BY distinct_albums_bought DESC;
""").df()


**Q8 的关键**：用 `COUNT(DISTINCT a.album_id)` 而不是 `COUNT(*)`，避免一首专辑里多首歌被买导致 album 重复计数。
**Q8's trick**: `COUNT(DISTINCT a.album_id)` not `COUNT(*)` — multiple tracks from the same album would otherwise inflate the album count.


<a id="14"></a>
## 14. 小结 / Summary

### 概念地图 / Concept map

```
JOIN
  │
  ├── INNER         ← 只保留两边都匹配的 / matching only
  │
  ├── LEFT  / RIGHT ← 一边全保留 / one-side preserved
  │     └── ON vs WHERE 陷阱 ⭐
  │
  ├── FULL          ← 两边全保留 / both preserved
  │
  ├── CROSS         ← 笛卡尔积 / |L| × |R|
  │
  └── SELF          ← 别名同表 / same table aliased
        └── 经常配 a.id < b.id 去重 + 排自己
```

### 💡 必背的"JOIN 三句箴言"

1. **`WHERE` 不要 = NULL** —— 永远用 `IS NULL` / `IS NOT NULL`
2. **LEFT JOIN 想保左表 unmatched 行** —— 右表条件放 **`ON`** 不要放 `WHERE`
3. **JOIN 前检查 key 唯一性**——重复 key → 笛卡尔爆炸

### 💡 Anti-join 三种写法的选择

| 写法 | 何时选 | 备注 |
|---|---|---|
| `NOT EXISTS` | **大多数情况** ⭐ | NULL 安全、SQL 优化器友好 |
| `LEFT JOIN ... WHERE IS NULL` | 想顺便取右表的列 | 易读 |
| `NOT IN` | 子查询保证无 NULL | 否则会"静默错"|

### 💡 工业速查 / Industry cheat sheet

```sql
-- 标准三表 JOIN 模板 / Standard 3-table template
SELECT a.x, b.y, c.z
FROM    a
JOIN    b ON a.k = b.k
JOIN    c ON b.k2 = c.k2
WHERE   a.filter > 0;

-- 安全的 anti-join / Safe anti-join
SELECT * FROM A
WHERE NOT EXISTS (SELECT 1 FROM B WHERE B.k = A.k);

-- 先聚合再 JOIN（避免乘起来）/ Aggregate before joining
FROM main
JOIN (SELECT id, COUNT(*) AS n FROM detail GROUP BY id) d USING (id);

-- 自连接找配对 / Self-join for pairs
FROM x a JOIN x b ON a.key = b.key AND a.id < b.id;
```

### 💡 面试速查 / Interview must-knows

1. **画 Venn 图解释 INNER vs LEFT** —— 闭眼能画
2. **LEFT JOIN + IS NULL = anti-join**
3. **重复 key → 行数爆炸**，用 `COUNT(DISTINCT)` 或预聚合救
4. **`ON` 条件 vs `WHERE` 条件 在 LEFT JOIN 里语义不同**
5. **`NOT IN` + NULL = 0 行**，用 `NOT EXISTS` 替

### 下一节预告 / Next up

**Part 1.3 · GROUP BY 与聚合** —— `GROUP BY`、`HAVING`、`ROLLUP` / `CUBE` / `GROUPING SETS`，再深入聚合的"分组宇宙"。
**Part 1.3 · GROUP BY & Aggregation** — into the multi-group aggregation universe with `HAVING`, `ROLLUP`, `CUBE`, `GROUPING SETS`.
